# Khojki: padding bug and normalization audit
Six matched 600-step pilots with unchanged labels. Isolate RGB red-border bug, freeze batch-normalization statistics, and freeze recurrent weights. Extend at most two promising pilots to 3000. Selection uses original 405-character development set. Previously viewed test sets are explicitly diagnostic. Save hashes and resumable checkpoints before disconnecting. Keep >=75 compute units.


In [1]:
from pathlib import Path
import sys,os,json,time,zipfile,hashlib,subprocess,shutil,datetime
BUNDLE=Path('/content/khojki-round4.zip');assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()=='48a70d7957b90929f205d661f1f38cecb4c6f509c913ac0a35573f054bc2436e'
ROOT=Path('/content/khojki-ocr')
with zipfile.ZipFile(BUNDLE) as z:z.extractall('/content')
for n,h in json.loads((ROOT/'cloud-manifest.json').read_text()).items():assert hashlib.sha256((ROOT/n).read_bytes()).hexdigest()==h,n
subprocess.run([sys.executable,'-m','pip','install','-q','freetype-py','uharfbuzz','scipy','opencv-python-headless','pytest','pypdf','pypdfium2'],check=True,timeout=180)
os.environ['PYTHONPATH']=str(ROOT/'src')
import torch
assert torch.cuda.is_available();print(torch.cuda.get_device_name(0))
ANN=ROOT/'data/round2/train-dev.json';EXP=ROOT/'experiments';RES=ROOT/'results';RES.mkdir(exist_ok=True)
def run(args,log,timeout=900):
 with log.open('w') as f:p=subprocess.run([sys.executable,*map(str,args)],stdout=f,stderr=subprocess.STDOUT,timeout=timeout)
 print(log.name,log.read_text()[-1300:]);p.check_returncode()
run(['-m','pytest',ROOT/'tests','-q'],RES/'round4-tests.log',120)


Tesla T4
round4-tests.log ................                                                         [100%]
16 passed in 3.67s



In [3]:
p=ROOT/'scripts/finetune.py'
s=p.read_text().replace('    if a.freeze_rnn:model.rnn.eval()\n','').replace('    for param in model.rnn.parameters():param.requires_grad_(False)','    for param in model.rnn.parameters():param.requires_grad_(False)\n    model.rnn.dropout=0.0')
p.write_text(s)
CASES={'r4-control':[],'r4-word-red':['--word-fraction','.4','--word-geometry','preserve','--legacy-rgb-word-padding'],'r4-word-white':['--word-fraction','.4','--word-geometry','preserve'],'r4-freezebn':['--freeze-bn'],'r4-word-white-freezebn':['--word-fraction','.4','--word-geometry','preserve','--freeze-bn'],'r4-freezernn':['--freeze-rnn']}
def train(name,steps):
 folder=EXP/name;folder.mkdir(exist_ok=True);local=folder/'last.pt';resume=local.exists();checkpoint=local if resume else EXP/'precision-packed/best.pt'
 if resume and torch.load(local,map_location='cpu',weights_only=False)['step']>=steps:print('Already complete',name,steps);return
 run([ROOT/'scripts/finetune.py','--checkpoint',checkpoint,'--annotations',ANN,'--steps',steps,'--lr','0.00005','--seed','503','--layout','robust','--bold-replay','--packed','--device','cuda','--output',name,*CASES[name],*(['--resume'] if resume else [])],EXP/(name+f'-{steps}.log'))
for name in CASES:train(name,600)
run([ROOT/'scripts/select_round4_model.py','--phase','pilot'],EXP/'round4-pilot.log')
report=json.loads((EXP/'round4-pilot-selection.json').read_text());eligible=sorted([r for r in report['experiments'] if r['name'] in CASES and r['errors']<=16],key=lambda r:r['errors'])[:2]
print('Extension rule: at most two pilots with <=16/405 edits; chosen',[r['name'] for r in eligible])
for row in eligible:train(row['name'],3000)
run([ROOT/'scripts/select_round4_model.py','--phase','final'],EXP/'round4-final.log')
# These already-viewed test sets are regression diagnostics, not new blind estimates.
for old in [2,3]:
 run([ROOT/'scripts/evaluate_pages.py','--checkpoint',ROOT/'models/round4-selected.pt','--annotations',ROOT/f'data/round{old}/test.json','--split','test','--device','cuda','--output',RES/f'round4-regression-r{old}'],RES/f'round4-regression-r{old}.log')


Already complete r4-control 600
Already complete r4-word-red 600
Already complete r4-word-white 600
Already complete r4-freezebn 600
Already complete r4-word-white-freezebn 600
r4-freezernn-600.log 79713, "validation_cer": 0.04691358024691358, "errors": 19, "characters": 405, "seconds": 18.67269936799994}
{"step": 425, "loss": 0.0006037223502062261, "seconds": 19.68360810799993}
{"step": 450, "loss": 0.01289614662528038, "seconds": 20.601238095999975}
{"step": 450, "loss": 0.01289614662528038, "validation_cer": 0.05185185185185185, "errors": 21, "characters": 405, "seconds": 20.871814936999954}
{"step": 475, "loss": 0.00187557318713516, "seconds": 21.812865257999988}
{"step": 500, "loss": 0.0009091857355087996, "seconds": 22.763444705999973}
{"step": 500, "loss": 0.0009091857355087996, "validation_cer": 0.056790123456790124, "errors": 23, "characters": 405, "seconds": 23.042806020999933}
{"step": 525, "loss": 0.001247276202775538, "seconds": 24.034668616999966}
{"step": 550, "loss": 0.

In [4]:
from IPython.display import display,HTML
import base64
EXPORTS=[]
for name in [*CASES,'round4-selection']:
 target=Path('/content')/(name+'-results.zip')
 with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
  paths=list((EXP/name).glob('*'))+list(EXP.glob(name+'-*.log')) if name!='round4-selection' else list(RES.rglob('*'))+list(EXP.glob('round4-*'))+[ROOT/'models/round4-selected.pt']
  for p in paths:
   if p.is_file():z.write(p,str(p.relative_to(ROOT)))
 display(HTML('<a download="'+target.name+'" href="data:application/zip;base64,'+base64.b64encode(target.read_bytes()).decode()+'">Download '+name+' results</a>'))
 EXPORTS.append({'name':target.name,'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()})
print('EXPORT_MANIFEST',json.dumps(EXPORTS))


Download r4-control results
Download r4-word-red results
Download r4-word-white results
Download r4-freezebn results
Download r4-word-white-freezebn results
Download r4-freezernn results
Download round4-selection results
EXPORT_MANIFEST [{"name": "r4-control-results.zip", "bytes": 23900474, "sha256": "62d9c3ea187c4d5f9112c6e74376b5478b0c5faf79c37eb52b3f430335f77c8d"}, {"name": "r4-word-red-results.zip", "bytes": 23869367, "sha256": "bff9a00be9083cc8a194b46a2705067d92cf59277a88393372106bd4e9886738"}, {"name": "r4-word-white-results.zip", "bytes": 23889467, "sha256": "808ef25b4772d052ba6932cec59d5d784104bfe5ac9f11d50a91d2dbcac8990a"}, {"name": "r4-freezebn-results.zip", "bytes": 23871944, "sha256": "969775dd1c268378f771a99e8d6b631e192e6110b4a7f343257339413fae4747"}, {"name": "r4-word-white-freezebn-results.zip", "bytes": 23841383, "sha256": "7b3993aa9aca1cb2ce57b69c9d5ba3277aec95bd12d39eea9352e1de49c1268c"}, {"name": "r4-freezernn-results.zip", "bytes": 12887417, "sha256": "0ef7ad2ef3dac

In [5]:
# Offline-prepared follow-up: matched low learning rate and source-audited label correction.
bundle=Path('/content/khojki-round4-followup.zip')
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='df4c43f08a52be96bddfb5434e979f1cb3188dbf49005231da5be00047733fca'
with zipfile.ZipFile(bundle) as z:z.extractall(ROOT)
run([ROOT/'scripts/round4_followup.py'],EXP/'round4-followup.log',2400)
run([ROOT/'scripts/evaluate_round4_final.py'],RES/'round4-final-evaluation.log',900)


round4-followup.log TRAINED r4-low-original 1000
TRAINED r4-low-corrected 1000
TRAINED r4-low-white 1000
DEV r4-low-original 16
DEV r4-low-corrected 18
DEV r4-low-white 18
DEV incumbent 14
DEV round4-selected 14
DEV r4-low-original 16
DEV r4-low-corrected 18
DEV r4-low-white 18
DEV low-mixed 18
DEV incumbent-weak 18
DEV incumbent-strong 16
DEV incumbent-blur 15
DEV incumbent-ink-average 16
DEV incumbent-blur-average 15
SELECTED {"name": "incumbent", "paths": ["models/round3-selected.pt"], "views": [[1.0, 0.0]], "errors": 14, "characters": 405}

round4-final-evaluation.log ": 0, "automatic_errors": 17, "checked_crop_errors": 17, "crop_changes_prediction_lines": 0, "invalid_inventory_characters": 0, "utf8_failures": 0, "automatic_cer": 0.1440677966101695, "checked_crop_cer": 0.1440677966101695, "net_crop_error_difference": 0}
{"lines": 5, "characters": 123, "missed_lines": 0, "automatic_errors": 26, "checked_crop_errors": 26, "crop_changes_prediction_lines": 0, "invalid_inventory_charact

In [6]:
EXPORTS_FOLLOWUP=[]
for name in ['r4-low-original','r4-low-corrected','r4-low-white','round4-followup']:
 target=Path('/content')/(name+'-results.zip')
 with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
  paths=list((EXP/name).glob('*'))+list(EXP.glob(name+'-*.log')) if name!='round4-followup' else list(RES.rglob('*'))+list(EXP.glob('round4-followup*'))+[ROOT/'models/round4-final.pt']
  for p in paths:
   if p.is_file():z.write(p,str(p.relative_to(ROOT)))
 display(HTML('<a download="'+target.name+'" href="data:application/zip;base64,'+base64.b64encode(target.read_bytes()).decode()+'">Download '+name+' results</a>'))
 EXPORTS_FOLLOWUP.append({'name':target.name,'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()})
print('FOLLOWUP_MANIFEST',json.dumps(EXPORTS_FOLLOWUP))


Download r4-low-original results
Download r4-low-corrected results
Download r4-low-white results
Download round4-followup results
FOLLOWUP_MANIFEST [{"name": "r4-low-original-results.zip", "bytes": 23879902, "sha256": "54c14406fdb3a4c7392cd58de777f0de9f8e5d5a9a262a3e1dbc06a8f7903806"}, {"name": "r4-low-corrected-results.zip", "bytes": 23876438, "sha256": "e935394d6a9727d6907c3a8a65abfb54a193e75fbde2e7553baa3b912041a290"}, {"name": "r4-low-white-results.zip", "bytes": 23879170, "sha256": "c9d6ebcc79d91c6b72c1dccd67255cf7cf789cd095a231b0deb9f51c53c35bc6"}, {"name": "round4-followup-results.zip", "bytes": 19686270, "sha256": "46da65ba48c373f588cf0b951439612a992b965e96d4e428142781bdc52bc2c3"}]


In [7]:
# Final deployment parity and CPU throughput, run after model selection/evaluation.
out=RES/'round4-cpu-demo'
run(['-m','khojki_ocr',ROOT/'data/pages/p21.png','--checkpoint',ROOT/'models/round4-final.pt','--output',out,'--device','cpu'],RES/'round4-cpu.log',180)
cpu=json.loads((out/'page-001/result.json').read_text());gpu=json.loads((RES/'round4-final-r4/p21/result.json').read_text());assert [r['text'] for r in cpu['lines']]==[r['text'] for r in gpu['lines']]
saved=torch.load(ROOT/'models/round4-final.pt',map_location='cpu',weights_only=False)
members=saved.get('ensemble_states') or [{'model':saved['model'],'packed':saved.get('packed',False)}]
verify={'cpu_gpu_page_text_equal':True,'cpu_page':{k:v for k,v in cpu.items() if k!='lines'},'cpu_run':json.loads((out/'run.json').read_text()),'members':len(members),'parameters':sum(sum(v.numel() for k,v in m['model'].items() if not any(s in k for s in ['running_mean','running_var','num_batches_tracked'])) for m in members),'views':saved.get('views',[[1.,0.]]),'checkpoint_sha256':hashlib.sha256((ROOT/'models/round4-final.pt').read_bytes()).hexdigest()}
(RES/'round4-verification.json').write_text(json.dumps(verify,indent=2));print('VERIFICATION',json.dumps(verify))
target=Path('/content/round4-verification-results.zip')
with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
 for p in [*out.rglob('*'),RES/'round4-verification.json',RES/'round4-cpu.log']:
  if p.is_file():z.write(p,str(p.relative_to(ROOT)))
display(HTML('<a download="'+target.name+'" href="data:application/zip;base64,'+base64.b64encode(target.read_bytes()).decode()+'">Download round4 verification</a>'))
print('VERIFY_MANIFEST',json.dumps([{'name':target.name,'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()}]))


round4-cpu.log Page 1: 16 lines; 3.28s; /content/khojki-ocr/results/round4-cpu-demo/page-001/review.html

VERIFICATION {"cpu_gpu_page_text_equal": true, "cpu_page": {"image_size": [760, 1200], "region": [58, 115, 694, 1119], "manual_region": false, "layout": "robust", "line_count": 16, "detection_seconds": 0.17774632999999085, "recognition_seconds": 2.7073083719994884, "processing_seconds": 3.2826206199999888, "scope": "Prototype for printed Khojki. All lines require review; handwriting is unvalidated."}, "cpu_run": {"source": "/content/khojki-ocr/data/pages/p21.png", "checkpoint_sha256": "b4398167ea03e71277d09eba4cf5a55ef3efac454738f649f01169343ec3abb6", "device": "cpu", "model_load_seconds": 0.04480118899982699, "pages": [{"page": 1, "image_size": [760, 1200], "region": [58, 115, 694, 1119], "manual_region": false, "layout": "robust", "line_count": 16, "detection_seconds": 0.17774632999999085, "recognition_seconds": 2.7073083719994884, "processing_seconds": 3.2826206199999888, "scope